# ***Mempersiapkan Dataset Hasil Scraping***

Notebook ini mempersiapkan kumpulan berita berbahasa Indonesia agar dapat digunakan dalam analisis teks atau klasifikasi dokumen. Data berasal dari dua kategori berita, yaitu **sport** dan **finance**. Setiap artikel akan diperiksa, dibersihkan, diubah menjadi representasi numerik, lalu direduksi dimensinya menggunakan Principal Component Analysis (PCA).

Alur pengolahan data pada notebook ini adalah:

1. memahami struktur dan kualitas dataset;
2. melakukan preprocessing teks;
3. membentuk matriks fitur dengan beberapa pendekatan; dan
4. mengurangi jumlah dimensi fitur menggunakan PCA.

Tujuan utama setiap tahap bukan sekadar menghasilkan tabel baru, tetapi mengubah teks yang belum terstruktur menjadi data numerik yang dapat diproses oleh algoritma machine learning.

## ***Data Understanding***

Data understanding adalah tahap untuk mengenali isi, bentuk, dan kondisi awal dataset sebelum dilakukan transformasi. Pada tahap ini, kita memastikan bahwa data yang digunakan memiliki kolom yang sesuai, jumlah kelas yang diharapkan, serta tidak memiliki masalah dasar seperti nilai kosong atau baris duplikat.

In [65]:
import pandas as pd
import re

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from tqdm import tqdm

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.decomposition import PCA

In [39]:
df_sport = pd.read_csv('data/dataset_detik_sport.csv')
df_finance = pd.read_csv('data/dataset_detik_finance.csv')

In [40]:
df_all_ori = pd.concat([df_sport, df_finance], ignore_index=True)

In [41]:
print(df_all_ori.tail())

    kategori                                                url  \
195  finance  https://finance.detik.com/industri/d-8648768/d...   
196  finance  https://finance.detik.com/bursa-dan-valas/d-86...   
197  finance  https://finance.detik.com/bursa-dan-valas/d-86...   
198  finance  https://finance.detik.com/infrastruktur/d-8648...   
199  finance  https://finance.detik.com/berita-ekonomi-bisni...   

                                                  teks  
195  PT Pupuk Indonesia (Persero) mencatatkan kiner...  
196  Indeks Harga Saham Gabungan (IHSG) ditutup mel...  
197  Bursa Efek Indonesia (BEI) menghadirkan instru...  
198  PT Brantas Abipraya (Persero) mulai memangkas ...  
199  Otorita Ibu Kota Nusantara (IKN) menjatuhkan s...  


In [42]:
df_all = df_all_ori.drop('url', axis=1)

In [43]:
print(df_all.head())

  kategori                                               teks
0    sport  Sean Gelael dan Team WRT 32 akan coba meraih p...
1    sport  KTM punya sejumlah alasan merekrut Luca Marini...
2    sport  MotoGP 2026 akan berlanjut ke San Marino. Seri...
3    sport  Cal Crutchlow membuat prediksi menarik soal ka...
4    sport  Team WRT 32 sudah menuntaskan sesi latihan beb...


In [44]:
print("Distribusi Kelas (Label) pada Dataset:")
print(df_all['kategori'].value_counts())

Distribusi Kelas (Label) pada Dataset:
kategori
sport      100
finance    100
Name: count, dtype: int64


In [45]:
print("Pengecekan data kosong (missing values) pada setiap kolom:")
print(df_all.isnull().sum())

Pengecekan data kosong (missing values) pada setiap kolom:
kategori    0
teks        0
dtype: int64


In [46]:
print("Pengecekan data duplikat (duplicate values) pada setiap kolom:")
print(df_all.duplicated().sum())

Pengecekan data duplikat (duplicate values) pada setiap kolom:
0


In [47]:
print("Mengecek panjang kata pada setiap baris data: ")
print(df_all['teks'].str.len())

Mengecek panjang kata pada setiap baris data: 
0      1136
1      2345
2      3599
3      2362
4      2163
       ... 
195    1665
196     930
197    3789
198    2721
199    1906
Name: teks, Length: 200, dtype: int64


## **Data Pre-Processing**

Teks berita masih berupa data tidak terstruktur. Kata yang sama dapat ditulis dengan perbedaan huruf besar, tanda baca, imbuhan, atau kata-kata umum yang tidak banyak membantu proses klasifikasi. Preprocessing bertujuan mengurangi variasi yang tidak diperlukan dan menghasilkan teks yang lebih konsisten.

Urutan pemrosesan pada notebook ini adalah:

1. **Pembersihan teks**: mengubah huruf menjadi lowercase, menghapus angka dan tanda baca, serta merapikan spasi.
2. **Stopword removal**: menghapus kata umum seperti kata penghubung yang memiliki nilai pembeda rendah.
3. **Stemming**: mengubah kata berimbuhan ke bentuk dasarnya menggunakan Sastrawi.

Kolom `teks`, `teks_bersih`, `teks_tanpa_stopword`, dan `teks_final` dipertahankan agar perubahan pada setiap tahap dapat dibandingkan.

In [48]:
# Inisialisasi Stemmer 
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Inisialisasi Bar Progress
tqdm.pandas()

# inisialisasi StopWordRemoverFactory dan StopWordRemover
factory_stopword = StopWordRemoverFactory()
stopword_remover = factory_stopword.create_stop_word_remover()

# Inisialisasi CountVectorizer dalam mode biner
vectorizer_biner = CountVectorizer(binary=True)

# Inisialisasi CountVectorizer (default untuk menghitung frekuensi/jumlah kata)
vectorizer_frekuensi = CountVectorizer()

# Inisialisasi TfidfTransformer
transformer_tfidf = TfidfTransformer()



In [49]:
def bersihkan_teks(teks):
    teks = str(teks).lower()
    teks = re.sub(r'\d+', '', teks) # Menghapus angka
    teks = re.sub(r'[^\w\s]', '', teks) # Menghapus tanda baca
    teks = re.sub(r'\s+', ' ', teks).strip() # Menghapus spasi berlebih
    return teks

In [50]:
# Inisialisasi Stemmer (ditaruh di luar fungsi agar tidak dipanggil berulang kali)
def normalisasi_teks(teks):
    return stemmer.stem(str(teks))

In [51]:
def hapus_stopword(teks):
    return stopword_remover.remove(str(teks))

In [52]:
print("1. Melakukan pembersihan teks (hapus angka, tanda baca, lowercase)...")
df_all['teks_bersih'] = df_all['teks'].progress_apply(bersihkan_teks)

1. Melakukan pembersihan teks (hapus angka, tanda baca, lowercase)...


100%|██████████| 200/200 [00:00<00:00, 4805.60it/s]


In [53]:
print("2. Menghapus stopword (kata-kata umum yang tidak memiliki makna penting)")
df_all['teks_tanpa_stopword'] = df_all['teks_bersih'].progress_apply(hapus_stopword)

2. Menghapus stopword (kata-kata umum yang tidak memiliki makna penting)


100%|██████████| 200/200 [00:00<00:00, 1952.59it/s]


In [54]:
print("3. Melakukan normalisasi (Stemming Sastrawi). Mohon tunggu beberapa saat...")
df_all['teks_final'] = df_all['teks_tanpa_stopword'].progress_apply(normalisasi_teks)

3. Melakukan normalisasi (Stemming Sastrawi). Mohon tunggu beberapa saat...


100%|██████████| 200/200 [06:14<00:00,  1.87s/it]


In [55]:

print("\n=== HASIL PRE-PROCESSING ===")
display(df_all[['kategori', 'teks', 'teks_bersih', 'teks_tanpa_stopword', 'teks_final']].head())


=== HASIL PRE-PROCESSING ===


,kategori,teks,teks_bersih,teks_tanpa_stopword,teks_final
0,sport,Sean Gelael dan Team WRT 32 akan coba meraih p...,sean gelael dan team wrt akan coba meraih podi...,sean gelael team wrt coba meraih podium lone s...,sean gelael team wrt coba raih podium lone sta...
1,sport,KTM punya sejumlah alasan merekrut Luca Marini...,ktm punya sejumlah alasan merekrut luca marini...,ktm punya sejumlah alasan merekrut luca marini...,ktm punya jumlah alas rekrut luca marini tech ...
2,sport,MotoGP 2026 akan berlanjut ke San Marino. Seri...,motogp akan berlanjut ke san marino seri ke mu...,motogp berlanjut san marino seri musim akan di...,motogp lanjut san marino seri musim akan gelar...
3,sport,Cal Crutchlow membuat prediksi menarik soal ka...,cal crutchlow membuat prediksi menarik soal ka...,cal crutchlow membuat prediksi menarik soal ka...,cal crutchlow buat prediksi tarik soal kandida...
4,sport,Team WRT 32 sudah menuntaskan sesi latihan beb...,team wrt sudah menuntaskan sesi latihan bebas ...,team wrt menuntaskan sesi latihan bebas jelang...,team wrt tuntas sesi latih bebas jelang balap ...


### Pembuatan Tabel 1: Binary Document-Term Matrix

Binary Document-Term Matrix menunjukkan apakah sebuah kata muncul dalam sebuah dokumen. Setiap baris mewakili satu artikel, sedangkan setiap kolom mewakili satu kata unik. Nilai yang digunakan hanya:

- `1`: kata muncul setidaknya satu kali dalam dokumen;
- `0`: kata tidak muncul dalam dokumen.

Representasi biner mengabaikan jumlah kemunculan kata. Karena itu, dua dokumen yang memuat kata yang sama tetap memiliki nilai yang sama meskipun frekuensinya berbeda. Kolom `label` disimpan terpisah sebagai target kategori dan bukan sebagai fitur teks.

In [56]:

# Transformasi teks_final menjadi matriks biner
matriks_biner = vectorizer_biner.fit_transform(df_all['teks_final'])

# Mendapatkan daftar kata unik untuk nama kolom
fitur_kata = vectorizer_biner.get_feature_names_out()

# Membuat DataFrame (Tabel 1) dari matriks
df_tabel1 = pd.DataFrame(matriks_biner.toarray(), columns=fitur_kata)

In [57]:
# Menambahkan label kategori di kolom paling akhir
df_tabel1['label'] = df_all['kategori'].values

# Tampilkan hasil Tabel 1
print("=== TABEL 1 (Eksistensi Kata: 0 atau 1) ===")
display(df_tabel1.tail())

=== TABEL 1 (Eksistensi Kata: 0 atau 1) ===


,aan,abadi,abai,abang,abangrangkasbitung,abdi,abdul,abimanyu,abipraya,abisabisan,...,zigmars,zon,zona,zone,zuffa,zulhas,zulkifli,zulverdi,zumba,zwinkle
195,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
196,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
197,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
198,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
199,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [58]:
print("Jumlah baris dan kolom pada Tabel 1:", df_tabel1.shape)

Jumlah baris dan kolom pada Tabel 1: (200, 5304)


### Pembuatan Tabel 2: Term Frequency Matrix / Bag of Words

Bag of Words merepresentasikan dokumen berdasarkan jumlah kemunculan setiap kata. Berbeda dari matriks biner, nilai pada tabel ini dapat lebih besar dari satu. Misalnya, jika sebuah kata muncul tiga kali dalam satu artikel, maka nilai fitur kata tersebut adalah `3`.

Metode ini sederhana dan dapat menangkap intensitas penggunaan kata, tetapi belum mempertimbangkan apakah kata tersebut juga sering muncul pada banyak dokumen lain. Matriks hasil pemrosesan teks biasanya memiliki banyak nilai nol sehingga secara konsep termasuk matriks berdimensi tinggi dan bersifat sparse.

In [59]:
# Transformasi teks_final menjadi matriks frekuensi
matriks_frekuensi = vectorizer_frekuensi.fit_transform(df_all['teks_final'])

# Membuat DataFrame (Tabel 2)
df_tabel2 = pd.DataFrame(matriks_frekuensi.toarray(), columns=vectorizer_frekuensi.get_feature_names_out())

In [60]:
# Menambahkan label kategori di kolom paling akhir
df_tabel2['label'] = df_all['kategori'].values

# Tampilkan hasil Tabel 2
print("=== TABEL 2 (Frekuensi Jumlah Kata) ===")
display(df_tabel2.tail())

=== TABEL 2 (Frekuensi Jumlah Kata) ===


,aan,abadi,abai,abang,abangrangkasbitung,abdi,abdul,abimanyu,abipraya,abisabisan,...,zigmars,zon,zona,zone,zuffa,zulhas,zulkifli,zulverdi,zumba,zwinkle
195,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
196,0,0,0,0,0,0,0,0,0,0,...,0,0,2,0,0,0,0,0,0,0
197,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
198,0,0,0,0,0,0,0,0,8,0,...,0,0,0,0,0,0,0,0,0,0
199,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [73]:
print("Jumlah baris dan kolom pada Tabel 2:", df_tabel2.shape)

Jumlah baris dan kolom pada Tabel 2: (200, 5304)


### Pembuatan Tabel 3: Term Frequency-Inverse Document Frequency (TF-IDF)

TF-IDF memberikan bobot lebih besar kepada kata yang penting bagi dokumen tertentu, tetapi tidak terlalu umum pada seluruh koleksi dokumen. Nilai ini menggabungkan dua komponen:

- **Term Frequency (TF)** mengukur seberapa sering kata muncul dalam sebuah dokumen.
- **Inverse Document Frequency (IDF)** mengurangi bobot kata yang muncul pada banyak dokumen.

Secara umum, perhitungannya dapat ditulis sebagai:

$$\mathrm{TFIDF}(t,d) = \mathrm{TF}(t,d) \times \mathrm{IDF}(t)$$

dengan:

$$\mathrm{IDF}(t) = \log\left(\frac{N}{\mathrm{df}(t)}\right)$$

`N` adalah jumlah dokumen dan `df(t)` adalah jumlah dokumen yang mengandung kata `t`. Kata yang sangat umum memperoleh bobot lebih rendah, sedangkan kata yang khas pada kategori tertentu cenderung memperoleh bobot lebih tinggi.

In [62]:
# Mengubah matriks frekuensi (dari Tabel 2) menjadi matriks TF-IDF
matriks_tfidf = transformer_tfidf.fit_transform(matriks_frekuensi)

# Membuat DataFrame (Tabel TF-IDF)
df_tfidf = pd.DataFrame(matriks_tfidf.toarray(), columns=vectorizer_frekuensi.get_feature_names_out())

In [64]:
# 4. Menambahkan label kategori di kolom paling akhir
df_tfidf['label'] = df_all['kategori'].values

# Tampilkan hasil Tabel TF-IDF
print("=== TABEL TF-IDF ===")
display(df_tfidf.tail())

=== TABEL TF-IDF ===


,aan,abadi,abai,abang,abangrangkasbitung,abdi,abdul,abimanyu,abipraya,abisabisan,...,zigmars,zon,zona,zone,zuffa,zulhas,zulkifli,zulverdi,zumba,zwinkle
195,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
196,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.163002,0.0,0.0,0.0,0.0,0.0,0.0,0.0
197,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
198,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.402394,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
199,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [66]:
dataset_sebelum_pca = df_tfidf.copy()

### Pembuatan Tabel 4: Principal Component Analysis (PCA)

Setelah TF-IDF, jumlah kolom dapat menjadi sangat besar karena setiap kata unik menjadi satu fitur. PCA digunakan untuk mereduksi dimensi tersebut menjadi sejumlah komponen baru yang lebih sedikit. Tujuannya adalah mempertahankan sebanyak mungkin variasi informasi dalam bentuk yang lebih ringkas.

Sebelum PCA, fitur dan label dipisahkan. Hanya fitur numerik yang boleh diproses PCA, sedangkan `label` dipertahankan agar kategori setiap artikel tetap dapat dikenali setelah reduksi. Komponen `PC1`, `PC2`, dan seterusnya bukan kata tertentu, melainkan kombinasi matematis dari banyak fitur kata.

Secara konseptual, PCA mencari arah baru dengan variansi terbesar. Komponen pertama mempertahankan variasi terbesar, komponen kedua mempertahankan variasi terbesar berikutnya dengan arah yang tegak lurus terhadap komponen pertama, dan seterusnya. Dengan menetapkan `n_components=5`, setiap dokumen direpresentasikan menggunakan lima nilai komponen.

PCA membantu mengurangi beban komputasi dan dapat memudahkan visualisasi atau pemodelan, tetapi sebagian informasi pasti dapat hilang. Oleh karena itu, jumlah komponen ideal sebaiknya ditentukan dengan mengevaluasi proporsi variance yang dipertahankan.

In [67]:
# Memisahkan fitur (kata unik) dan label ('sport'/'finance')
fitur = dataset_sebelum_pca.drop('label', axis=1)
label = dataset_sebelum_pca['label']

In [68]:
# 2. Inisialisasi PCA 
# (Saya set n_components=5 sebagai nilai awal)
pca = PCA(n_components=5)

# Melakukan reduksi dimensi
fitur_pca = pca.fit_transform(fitur)

In [69]:
# 3. Variabel penampungan SESUDAH reduksi
kolom_pca = [f'PC{i+1}' for i in range(fitur_pca.shape[1])]
dataset_sesudah_pca = pd.DataFrame(fitur_pca, columns=kolom_pca)
dataset_sesudah_pca['label'] = label.values

In [70]:
# Tampilkan hasil
print(f"Jumlah kolom sebelum PCA : {dataset_sebelum_pca.shape[1]}")
print(f"Jumlah kolom sesudah PCA : {dataset_sesudah_pca.shape[1]}\n")

Jumlah kolom sebelum PCA : 5304
Jumlah kolom sesudah PCA : 6



Angka 5 hanyalah nilai placeholder (sementara) untuk mendemonstrasikan bahwa kode berhasil mereduksi ribuan kolom kata unik menjadi format yang jauh lebih kecil. Tidak ada alasan matematis mutlak di balik angka 5 pada kasus Anda saat ini.

Dalam pengerjaan tugas analisis data teks yang sebenarnya, penentuan parameter n_components umumnya didasarkan pada salah satu dari tiga tujuan berikut:

* Kebutuhan Visualisasi: Gunakan n_components=2 atau n_components=3. Ini akan memampatkan data ke dalam sumbu X, Y, dan Z sehingga posisi setiap dokumen berita (sport vs finance) dapat digambar dan dianalisis menggunakan scatter plot.

* Mempertahankan Informasi (Explained Variance): Gunakan nilai float/desimal, misalnya n_components=0.85. PCA secara otomatis akan menghitung dan mempertahankan jumlah dimensi (kolom) yang diperlukan untuk merangkum 85% dari total varians (informasi asli) data TF-IDF Anda.

* Optimalisasi Algoritma: Menggunakan teknik Grid Search untuk mencoba berbagai angka (misal 50, 100, atau 200) demi mencari jumlah dimensi yang menghasilkan akurasi tertinggi pada model machine learning selanjutnya.

In [72]:
print("=== DATASET SESUDAH PCA ===")
display(dataset_sesudah_pca.tail())

=== DATASET SESUDAH PCA ===


,PC1,PC2,PC3,PC4,PC5,label
195,-0.117623,-0.000077,-0.166872,-0.330100,0.085351,finance
196,-0.050863,0.018440,-0.022229,-0.027375,0.002124,finance
197,-0.103782,-0.055460,-0.031896,-0.064615,0.018911,finance
198,-0.083203,0.039178,-0.096859,-0.154276,0.028280,finance
199,-0.071497,0.055870,-0.044271,-0.067967,0.020659,finance
